# Eurostat & OECD Transport Data Analysis

This notebook analyzes Eurostat and OECD transport datasets with optimized functions for handling official statistical data formats.

## Features
- Automatic detection and loading of Eurostat/OECD CSV formats
- Multi-file batch processing
- Flag and metadata interpretation
- Time series analysis and visualization
- Cross-country and cross-dataset comparisons
- Memory-efficient data handling

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from typing import Dict, List, Tuple

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) > 0.01 else f'{x:.2e}')

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Data Loading Functions

Custom functions to handle Eurostat and OECD data formats

In [ ]:
def load_eurostat_csv(filepath: str) -> pd.DataFrame:
    """
    Load Eurostat CSV file with proper data type handling and flag interpretation.
    
    Args:
        filepath: Path to the Eurostat CSV file
        
    Returns:
        pd.DataFrame: Loaded and processed dataframe
    """
    df = pd.read_csv(filepath, low_memory=False)
    
    # Convert TIME_PERIOD to datetime if it exists
    if 'TIME_PERIOD' in df.columns:
        df['TIME_PERIOD'] = pd.to_datetime(df['TIME_PERIOD'], format='%Y', errors='coerce')
    
    # Convert OBS_VALUE to numeric, coercing errors to NaN
    if 'OBS_VALUE' in df.columns:
        df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    
    # Optimize memory by converting object columns to category where appropriate
    for col in df.columns:
        if df[col].dtype == 'object' and col not in ['LAST UPDATE']:
            num_unique = df[col].nunique()
            if num_unique / len(df) < 0.5:  # If less than 50% unique values
                df[col] = df[col].astype('category')
    
    return df


def load_oecd_csv(filepath: str) -> pd.DataFrame:
    """
    Load OECD CSV file handling the double-header format.
    
    Args:
        filepath: Path to the OECD CSV file
        
    Returns:
        pd.DataFrame: Loaded and processed dataframe
    """
    # Read the file to detect format
    with open(filepath, 'r') as f:
        first_line = f.readline()
    
    # OECD files often have metadata in first row
    if 'STRUCTURE' in first_line:
        # Skip the metadata row and use the actual header
        df = pd.read_csv(filepath, skiprows=1, low_memory=False)
    else:
        df = pd.read_csv(filepath, low_memory=False)
    
    # Convert TIME_PERIOD to datetime if it exists
    if 'TIME_PERIOD' in df.columns:
        df['TIME_PERIOD'] = pd.to_datetime(df['TIME_PERIOD'], format='%Y', errors='coerce')
    elif 'Time period' in df.columns:
        df['TIME_PERIOD'] = pd.to_datetime(df['Time period'], format='%Y', errors='coerce')
    
    # Convert OBS_VALUE to numeric
    if 'OBS_VALUE' in df.columns:
        df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    elif 'Observation value' in df.columns:
        df['OBS_VALUE'] = pd.to_numeric(df['Observation value'], errors='coerce')
    
    # Optimize memory
    for col in df.columns:
        if df[col].dtype == 'object':
            num_unique = df[col].nunique()
            if num_unique / len(df) < 0.5:
                df[col] = df[col].astype('category')
    
    return df


def load_all_datasets(directory: str = '.') -> Dict[str, pd.DataFrame]:
    """
    Load all CSV files from a directory.
    
    Args:
        directory: Directory containing CSV files
        
    Returns:
        Dictionary mapping filenames to dataframes
    """
    datasets = {}
    csv_files = list(Path(directory).glob('*.csv'))
    
    print(f'Found {len(csv_files)} CSV files\n')
    
    for filepath in csv_files:
        filename = filepath.name
        print(f'Loading {filename}...', end=' ')
        
        try:
            if 'OECD' in filename:
                df = load_oecd_csv(str(filepath))
            else:
                df = load_eurostat_csv(str(filepath))
            
            datasets[filename] = df
            print(f'✓ Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
        except Exception as e:
            print(f'✗ Error: {str(e)}')
    
    return datasets


def interpret_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add human-readable interpretation of OBS_FLAG values.
    
    Common Eurostat flags:
    - b: break in time series
    - e: estimated
    - p: provisional
    - u: low reliability
    - c: confidential
    - m: missing/not available
    """
    flag_meanings = {
        'b': 'Break in time series',
        'B': 'Break in time series',
        'e': 'Estimated',
        'p': 'Provisional',
        'P': 'Provisional',
        'u': 'Low reliability',
        'c': 'Confidential',
        'm': 'Missing/Not available',
        'd': 'Definition differs'
    }
    
    if 'OBS_FLAG' in df.columns:
        df['FLAG_MEANING'] = df['OBS_FLAG'].map(flag_meanings)
    
    if 'OBS_STATUS' in df.columns:
        df['STATUS_MEANING'] = df['OBS_STATUS'].map(flag_meanings)
    
    return df

print('Data loading functions defined successfully!')

## 3. Load Datasets

In [ ]:
# Load all CSV files in the current directory
datasets = load_all_datasets()

print(f'\n{"="*60}')
print(f'Successfully loaded {len(datasets)} datasets')
print(f'{"="*60}')

## 4. Dataset Overview

In [ ]:
# Overview of all datasets
overview_data = []

for name, df in datasets.items():
    overview_data.append({
        'Dataset': name[:50],  # Truncate long names
        'Rows': f'{df.shape[0]:,}',
        'Columns': df.shape[1],
        'Memory (MB)': f'{df.memory_usage(deep=True).sum() / 1024**2:.2f}',
        'Time Range': f"{df['TIME_PERIOD'].min().year if 'TIME_PERIOD' in df.columns and not df['TIME_PERIOD'].isna().all() else 'N/A'} - {df['TIME_PERIOD'].max().year if 'TIME_PERIOD' in df.columns and not df['TIME_PERIOD'].isna().all() else 'N/A'}",
        'Countries': df['geo'].nunique() if 'geo' in df.columns else (df['REF_AREA'].nunique() if 'REF_AREA' in df.columns else 'N/A')
    })

overview_df = pd.DataFrame(overview_data)
display(overview_df)

## 5. Detailed Analysis of Individual Datasets

In [ ]:
# Select a dataset for detailed analysis (change as needed)
# By default, select the first dataset
selected_dataset_name = list(datasets.keys())[0]
df = datasets[selected_dataset_name].copy()

# Add flag interpretations
df = interpret_flags(df)

print(f'Analyzing: {selected_dataset_name}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n')

# Display first few rows
print('First 5 rows:')
display(df.head())

# Display column information
print('\nColumn Information:')
df.info()

In [ ]:
# Statistical summary of numerical columns
print('Statistical Summary:')
display(df.describe())

In [ ]:
# Check data quality
print('Data Quality Report:\n')

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)

print('Missing Values:')
display(missing_df[missing_df['Missing Count'] > 0])

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'\nDuplicate rows: {duplicates:,}')

# Flag distribution (if applicable)
if 'OBS_FLAG' in df.columns:
    print('\nObservation Flags Distribution:')
    flag_counts = df['OBS_FLAG'].value_counts(dropna=False)
    if 'FLAG_MEANING' in df.columns:
        flag_summary = pd.DataFrame({
            'Flag': flag_counts.index,
            'Count': flag_counts.values,
            'Percentage': (flag_counts.values / len(df)) * 100,
            'Meaning': df.groupby('OBS_FLAG')['FLAG_MEANING'].first()[flag_counts.index].values
        })
    else:
        flag_summary = pd.DataFrame({
            'Flag': flag_counts.index,
            'Count': flag_counts.values,
            'Percentage': (flag_counts.values / len(df)) * 100
        })
    display(flag_summary)

## 6. Geographical Analysis

In [ ]:
# Country/region analysis
geo_col = 'geo' if 'geo' in df.columns else ('REF_AREA' if 'REF_AREA' in df.columns else None)

if geo_col and 'OBS_VALUE' in df.columns:
    print(f'Geographical Analysis (by {geo_col}):\n')
    
    # Statistics by country
    geo_stats = df.groupby(geo_col).agg({
        'OBS_VALUE': ['count', 'mean', 'sum', 'min', 'max']
    }).round(2)
    geo_stats.columns = ['Observations', 'Mean', 'Total', 'Min', 'Max']
    geo_stats = geo_stats.sort_values('Total', ascending=False)
    
    print('Top 10 Countries/Regions by Total Value:')
    display(geo_stats.head(10))
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top countries by total value
    top_n = min(15, len(geo_stats))
    geo_stats.head(top_n)['Total'].plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_xlabel('Total Value')
    axes[0].set_ylabel('Country/Region')
    axes[0].set_title(f'Top {top_n} Regions by Total Value')
    axes[0].grid(True, alpha=0.3)
    
    # Number of observations per country
    geo_stats.head(top_n)['Observations'].plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_xlabel('Number of Observations')
    axes[1].set_ylabel('Country/Region')
    axes[1].set_title(f'Top {top_n} Regions by Data Availability')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('No geographical column found for analysis.')

## 7. Time Series Analysis

In [ ]:
# Time series analysis
if 'TIME_PERIOD' in df.columns and 'OBS_VALUE' in df.columns:
    print('Time Series Analysis:\n')
    
    # Overall trend
    ts_data = df.groupby('TIME_PERIOD')['OBS_VALUE'].agg(['mean', 'sum', 'count'])
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Mean value over time
    axes[0].plot(ts_data.index, ts_data['mean'], marker='o', linewidth=2, markersize=4)
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Mean Value')
    axes[0].set_title('Mean Value Over Time')
    axes[0].grid(True, alpha=0.3)
    
    # Total value over time
    axes[1].plot(ts_data.index, ts_data['sum'], marker='o', linewidth=2, markersize=4, color='green')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Total Value')
    axes[1].set_title('Total Value Over Time')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate year-over-year growth
    ts_data['yoy_growth'] = ts_data['sum'].pct_change() * 100
    print('Year-over-Year Growth Rate (%):')
    print(ts_data[['sum', 'yoy_growth']].tail(10))
else:
    print('Time period data not available for time series analysis.')

In [ ]:
# Time series by country (for selected major countries)
if 'TIME_PERIOD' in df.columns and geo_col and 'OBS_VALUE' in df.columns:
    print('Time Series Comparison by Country:\n')
    
    # Select top countries by total value
    top_countries = df.groupby(geo_col)['OBS_VALUE'].sum().nlargest(6).index
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for country in top_countries:
        country_data = df[df[geo_col] == country].groupby('TIME_PERIOD')['OBS_VALUE'].sum()
        ax.plot(country_data.index, country_data.values, marker='o', linewidth=2, 
                markersize=4, label=country, alpha=0.8)
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Value', fontsize=12)
    ax.set_title('Time Series Comparison - Top 6 Countries', fontsize=14, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Category Analysis

Analyze different categories/dimensions in the dataset

In [ ]:
# Identify categorical columns (excluding metadata columns)
exclude_cols = ['DATAFLOW', 'LAST UPDATE', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 
                'CONF_STATUS', 'FLAG_MEANING', 'STATUS_MEANING', geo_col]
categorical_cols = [col for col in df.select_dtypes(include=['object', 'category']).columns 
                   if col not in exclude_cols and df[col].nunique() < 50]

if categorical_cols:
    print(f'Found {len(categorical_cols)} categorical dimension(s) for analysis:\n')
    
    for col in categorical_cols:
        print(f'\n{"="*60}')
        print(f'Category: {col}')
        print(f'{"="*60}')
        
        # Value counts
        value_counts = df[col].value_counts()
        print(f'\nUnique values: {len(value_counts)}')
        display(value_counts.to_frame('Count'))
        
        # Analysis by category if we have numerical values
        if 'OBS_VALUE' in df.columns:
            category_stats = df.groupby(col)['OBS_VALUE'].agg(['count', 'mean', 'sum']).round(2)
            category_stats.columns = ['Observations', 'Mean', 'Total']
            category_stats = category_stats.sort_values('Total', ascending=False)
            
            print(f'\nStatistics by {col}:')
            display(category_stats)
            
            # Visualization
            if len(value_counts) <= 20:  # Only plot if not too many categories
                fig, axes = plt.subplots(1, 2, figsize=(16, 6))
                
                # Bar plot of totals
                category_stats['Total'].plot(kind='barh', ax=axes[0], color='steelblue')
                axes[0].set_xlabel('Total Value')
                axes[0].set_ylabel(col)
                axes[0].set_title(f'Total Value by {col}')
                axes[0].grid(True, alpha=0.3)
                
                # Bar plot of means
                category_stats['Mean'].plot(kind='barh', ax=axes[1], color='coral')
                axes[1].set_xlabel('Mean Value')
                axes[1].set_ylabel(col)
                axes[1].set_title(f'Mean Value by {col}')
                axes[1].grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
else:
    print('No categorical dimensions found for analysis.')

## 9. Cross-Dataset Comparison

Compare trends across multiple datasets

In [ ]:
# Compare time series across all datasets
if len(datasets) > 1:
    print('Cross-Dataset Comparison:\n')
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    comparison_data = []
    
    for name, dataset in datasets.items():
        if 'TIME_PERIOD' in dataset.columns and 'OBS_VALUE' in dataset.columns:
            ts = dataset.groupby('TIME_PERIOD')['OBS_VALUE'].sum()
            
            # Normalize for comparison (to 0-100 scale)
            if ts.max() > 0:
                ts_normalized = (ts / ts.max()) * 100
                ax.plot(ts_normalized.index, ts_normalized.values, 
                       marker='o', linewidth=2, markersize=3, 
                       label=name[:40], alpha=0.7)  # Truncate long names
            
            comparison_data.append({
                'Dataset': name[:40],
                'Time Range': f"{ts.index.min().year} - {ts.index.max().year}",
                'Data Points': len(ts),
                'Total Value': f'{ts.sum():,.0f}',
                'Mean Value': f'{ts.mean():,.2f}'
            })
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Normalized Value (% of maximum)', fontsize=12)
    ax.set_title('Cross-Dataset Time Series Comparison (Normalized)', fontsize=14, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print('\nDataset Comparison Summary:')
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df)
else:
    print('Only one dataset loaded. Load multiple datasets for cross-comparison.')

## 10. Data Export Functions

In [ ]:
def export_analysis_results(df: pd.DataFrame, geo_col: str = None, prefix: str = 'analysis'):
    """
    Export key analysis results to CSV files.
    
    Args:
        df: DataFrame to analyze
        geo_col: Name of geography column
        prefix: Prefix for output filenames
    """
    # Time series summary
    if 'TIME_PERIOD' in df.columns and 'OBS_VALUE' in df.columns:
        ts_summary = df.groupby('TIME_PERIOD')['OBS_VALUE'].agg(['count', 'mean', 'sum'])
        ts_summary.to_csv(f'{prefix}_time_series.csv')
        print(f'✓ Exported: {prefix}_time_series.csv')
    
    # Geographic summary
    if geo_col and 'OBS_VALUE' in df.columns:
        geo_summary = df.groupby(geo_col)['OBS_VALUE'].agg(['count', 'mean', 'sum'])
        geo_summary.to_csv(f'{prefix}_geographic.csv')
        print(f'✓ Exported: {prefix}_geographic.csv')
    
    # Cleaned full dataset
    df_clean = df.dropna(subset=['OBS_VALUE'])
    df_clean.to_csv(f'{prefix}_cleaned.csv', index=False)
    print(f'✓ Exported: {prefix}_cleaned.csv')

# Example usage (uncomment to use):
# export_analysis_results(df, geo_col, 'transport_analysis')

print('Export functions ready to use!')

## 11. Summary Report

In [ ]:
# Generate comprehensive summary report
print('='*70)
print('TRANSPORT DATA ANALYSIS SUMMARY REPORT')
print('='*70)

print(f'\n📊 DATASETS LOADED: {len(datasets)}')
print(f'\nTotal observations across all datasets: {sum(d.shape[0] for d in datasets.values()):,}')
print(f'Total memory usage: {sum(d.memory_usage(deep=True).sum() for d in datasets.values()) / 1024**2:.2f} MB')

if datasets:
    print(f'\n📁 DATASET DETAILS:')
    for name, dataset in datasets.items():
        print(f'\n  • {name}')
        print(f'    - Rows: {dataset.shape[0]:,}')
        print(f'    - Columns: {dataset.shape[1]}')
        
        if 'TIME_PERIOD' in dataset.columns:
            valid_dates = dataset['TIME_PERIOD'].dropna()
            if len(valid_dates) > 0:
                print(f'    - Time range: {valid_dates.min().year} - {valid_dates.max().year}')
        
        geo_column = 'geo' if 'geo' in dataset.columns else ('REF_AREA' if 'REF_AREA' in dataset.columns else None)
        if geo_column:
            print(f'    - Countries/Regions: {dataset[geo_column].nunique()}')
        
        if 'OBS_VALUE' in dataset.columns:
            valid_obs = dataset['OBS_VALUE'].dropna()
            if len(valid_obs) > 0:
                print(f'    - Valid observations: {len(valid_obs):,}')
                print(f'    - Value range: {valid_obs.min():,.0f} - {valid_obs.max():,.0f}')

print(f'\n{"="*70}')
print('✓ Analysis complete!')
print(f'{"="*70}')

## 12. Next Steps & Recommendations

Based on this analysis, consider:

1. **Data Quality Improvements**:
   - Handle missing data and flags appropriately
   - Investigate breaks in time series
   - Validate estimated and provisional values

2. **Further Analysis**:
   - Perform statistical tests (e.g., trend analysis, seasonality)
   - Build forecasting models
   - Compare performance across countries/regions
   - Analyze relationships between different transport modes

3. **Visualization**:
   - Create interactive dashboards (e.g., using Plotly)
   - Generate maps for geographical analysis
   - Build comparison reports

4. **Data Integration**:
   - Merge related datasets for comprehensive analysis
   - Add external data sources (e.g., economic indicators)
   - Create derived metrics and indicators